# Classificacao local com LM Studio

Este notebook classifica os posts em `sentinel_replica_jsons_cleaned_eval/` usando o mesmo padrao de chamada do notebook de referencia `example_local.ipynb`: `lmstudio.AsyncClient`, `model.respond(...)` e `response_format` com `pydantic`.

In [1]:
from __future__ import annotations

import asyncio
import json
from pathlib import Path
from typing import Any

import lmstudio as lms
import pandas as pd
from pydantic import BaseModel
from tqdm.asyncio import tqdm

INPUT_DIR = Path("../sentinel_replica_jsons_cleaned_eval")
OUTPUT_DIR = Path("../sentinel_replica_jsons_classified_eval")
OUTPUT_DIR.mkdir(exist_ok=True)

MODEL = "qwen/qwen3.5-9b"
LMS_HOST = "172.19.160.1:1234"
MAX_CONCURRENT = 16

JSON_FILES = sorted(INPUT_DIR.glob("*.json"))
len(JSON_FILES), JSON_FILES[:3]

(12,
 [PosixPath('../sentinel_replica_jsons_cleaned_eval/HackingBlogsGroup.json'),
  PosixPath('../sentinel_replica_jsons_cleaned_eval/PHOfficial.json'),
  PosixPath('../sentinel_replica_jsons_cleaned_eval/WokeIntelDrops.json')])

In [2]:
from typing import Any, Literal
from pydantic import BaseModel, ConfigDict, Field
import json
import re
from urllib.parse import urlsplit

# Códigos compactos usados apenas no I/O da LLM.
THREAT_CODE_TO_LABEL = {
    "n": "not_a_threat",
    "mw": "malware",
    "rw": "ransomware",
    "ph": "phishing",
    "cred": "credential_theft",
    "scam": "fraud_or_scam",
    "vuln": "vulnerability_or_exploit",
    "leak": "data_breach_or_leak",
    "ato": "account_takeover",
    "ddos": "ddos_or_disruption",
    "sc": "supply_chain_compromise",
    "ins": "insider_threat",
    "c2": "botnet_or_c2",
    "ia": "initial_access_activity",
    "recon": "reconnaissance",
    "pe": "privilege_escalation",
    "lm": "lateral_movement",
    "exfil": "exfiltration",
    "wipe": "wiper_or_destruction",
    "disinfo": "disinformation_or_influence",
    "hybrid": "physical_or_hybrid_threat",
    "other": "other_cyber",
}

SEV_CODE_TO_LABEL = {
    "i": "informational",
    "l": "low",
    "m": "medium",
    "h": "high",
    "c": "critical",
}

CONF_CODE_TO_LABEL = {
    "l": "low",
    "m": "medium",
    "h": "high",
}

TARGET_CODE_TO_LABEL = {
    "ind": "individual",
    "org": "company",
    "gov": "government",
    "ci": "critical_infrastructure",
    "u": "unknown",
}

ACTOR_TYPE_CODE_TO_LABEL = {
    "crim": "criminal",
    "state": "state",
    "hack": "hacktivist",
    "ins": "insider",
    "u": "unknown",
}


class LLMEntity(BaseModel):
    model_config = ConfigDict(extra="ignore")

    # Apenas entidades contextuais que regex não pega bem.
    t: Literal[
        "actor",
        "malware",
        "tool",
        "product",
        "vuln",
        "victim",
        "sector",
        "country",
        "ttp",
    ]
    v: str


class CompactThreatClassification(BaseModel):
    model_config = ConfigDict(extra="ignore")

    # relevant
    r: bool = False

    # threat_label
    tl: Literal[
        "n", "mw", "rw", "ph", "cred", "scam", "vuln", "leak", "ato",
        "ddos", "sc", "ins", "c2", "ia", "recon", "pe", "lm", "exfil",
        "wipe", "disinfo", "hybrid", "other"
    ] = "n"

    # secondary_labels
    sl: list[str] = Field(default_factory=list)

    # severity, confidence
    sv: Literal["i", "l", "m", "h", "c"] = "i"
    cf: Literal["l", "m", "h"] = "m"

    # target_type, actor_type
    tt: Literal["ind", "org", "gov", "ci", "u"] = "u"
    at: Literal["crim", "state", "hack", "ins", "u"] = "u"

    # correlation fields compactos
    v: str = ""   # victim_name
    a: str = ""   # threat_actor_name
    mw: str = ""  # malware/tool family if central
    d: str = ""   # incident_date YYYY-MM-DD only if explicit

    # graph contextual entities
    e: list[LLMEntity] = Field(default_factory=list)

    # human review
    hr: bool = False

In [3]:
SYSTEM_PROMPT = (
    "CTI classifier. Output only minified valid JSON. "
    "No markdown. Do not invent unknown fields."
)

LABEL_HINT = (
    "tl/sl codes: n=not cyber threat, mw=malware, rw=ransomware, ph=phishing, "
    "cred=credential theft, scam=fraud/scam, vuln=vulnerability/exploit, "
    "leak=data breach/leak, ato=account takeover, ddos=disruption, "
    "sc=supply chain, ins=insider, c2=botnet/c2, ia=initial access, "
    "recon=recon, pe=priv esc, lm=lateral movement, exfil=exfiltration, "
    "wipe=wiper/destruction, disinfo=influence, hybrid=physical/hybrid, other=other cyber."
)

OUTPUT_HINT = (
    'JSON keys: r,tl,sl,sv,cf,tt,at,v,a,mw,d,e,hr. '
    'sv=i|l|m|h|c; cf=l|m|h; tt=ind|org|gov|ci|u; at=crim|state|hack|ins|u. '
    'v=primary victim name or empty. '
    'e=[{"t":"actor|malware|tool|product|vuln|victim|sector|country|ttp","v":"name"}].'
)

CLASSIFICATION_RULES = (
    "Rules: r=true for cyber incidents, vulnerabilities, malware, phishing, breaches, "
    "exploits, attack campaigns, defensive alerts, or CTI news. "
    "Use tl=n only for clearly non-cyber or generic promotional content. "
    "Prefer tl=leak for data exposure/breach, tl=vuln for CVE/exploit, "
    "tl=mw for malware/infostealer, tl=ph for phishing, tl=rw for ransomware. "
    "v is only a canonical named victim organization/person/group; leave empty for generic "
    "users, sectors, affected vendors/products, CVEs, severity words, or unknown victims. "
    "Put affected software/hardware/platforms in e as product, not in v. "
    "Leave a/mw/d empty if unknown. d only if incident date is explicit YYYY-MM-DD. "
    "e should include semantic context names: victims, actors, malware, tools, products, "
    "sectors, countries, TTPs. Do not repeat IP/domain/url/hash/email/wallet/handle in e; "
    "regex extracts them. Never copy enum hints like i|l|m|h|c into any field. "
    "Canonical names only; no actor version suffix."
)


def build_prompt(post: dict[str, Any]) -> str:
    # Para economizar tokens: envie só normalized se existir.
    text = (
        post.get("cleaning", {}).get("normalized_message")
        or post.get("message", "")
        or ""
    )
    text = str(text).strip()

    return (
        f"{LABEL_HINT}\n"
        f"{OUTPUT_HINT}\n"
        f"{CLASSIFICATION_RULES}\n"
        f"MSG:\n{text}"
    )

In [4]:
URL_RE = re.compile(r"https?://[^\s<>\")\]]+", re.I)
EMAIL_RE = re.compile(r"\b[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,}\b", re.I)
IP_RE = re.compile(r"\b(?:\d{1,3}\.){3}\d{1,3}\b")
CVE_RE = re.compile(r"\bCVE-\d{4}-\d{4,7}\b", re.I)
HASH_RE = re.compile(r"\b[a-fA-F0-9]{32}\b|\b[a-fA-F0-9]{40}\b|\b[a-fA-F0-9]{64}\b")
ETH_RE = re.compile(r"\b0x[a-fA-F0-9]{40}\b")
BTC_RE = re.compile(r"\b(?:bc1|[13])[a-zA-HJ-NP-Z0-9]{25,62}\b")
HANDLE_RE = re.compile(r"(?<![\w@])@[A-Za-z0-9_]{3,32}\b")

DOMAIN_RE = re.compile(
    r"\b(?:[a-z0-9](?:[a-z0-9-]{0,61}[a-z0-9])?\.)+[a-z]{2,}\b",
    re.I,
)
DEFANGED_DOMAIN_RE = re.compile(
    r"\b(?:[a-z0-9](?:[a-z0-9-]{0,61}[a-z0-9])?(?:\.|\[\.\]|\(\.\)))+[a-z]{2,}\b",
    re.I,
)

_SOCIAL_DOMAINS = {
    "linkedin.com", "lnkd.in", "twitter.com", "x.com", "t.me", "telegram.me",
    "youtube.com", "youtu.be", "facebook.com", "instagram.com", "tiktok.com",
    "github.com", "github.io", "udemy.com", "coursera.org", "udacity.com",
    "medium.com", "whatsapp.com",
}

_COMMON_TLDS = {
    "app", "br", "biz", "cloud", "cn", "co", "com", "dev", "edu", "gov", "info",
    "io", "ir", "me", "mil", "net", "news", "org", "pro", "ru", "run", "site",
    "tech", "to", "top", "uk", "us", "xyz",
}

_ENTITY_TYPE_TO_OUTPUT = {
    "ip": "ip_address",
    "url": "url",
    "domain": "domain",
    "email": "email",
    "cve": "cve",
    "hash": "hash",
    "wallet": "wallet",
    "handle": "handle",
    "actor": "threat_actor",
    "malware": "malware",
    "tool": "tool",
    "product": "product",
    "vuln": "vulnerability",
    "victim": "victim",
    "sector": "sector",
    "country": "country",
    "ttp": "ttp",
}

_IOC_ENTITY_TYPES = {
    "ip_address", "domain", "url", "hash", "email", "wallet", "handle", "cve"
}


def _clean_regex_value(value: str) -> str:
    return value.strip().rstrip(".,;:)]}")


def _refang_domain(value: str) -> str:
    return value.replace("[.]", ".").replace("(.)", ".")


def _strip_www(domain: str) -> str:
    domain = domain.lower()
    return domain[4:] if domain.startswith("www.") else domain


def _url_domain(url: str) -> str:
    try:
        return _strip_www(urlsplit(url).netloc)
    except ValueError:
        return ""


def _looks_like_domain(value: str) -> bool:
    domain = _strip_www(_refang_domain(value).rstrip("."))
    parts = domain.rsplit(".", 1)
    return len(parts) == 2 and parts[1] in _COMMON_TLDS


def _valid_source_url(url: str) -> str:
    url = _clean_regex_value(url)
    if not url.startswith("http"):
        return ""
    netloc = _url_domain(url)
    if not netloc:
        return ""
    if any(netloc == d or netloc.endswith("." + d) for d in _SOCIAL_DOMAINS):
        return ""
    return url


def _entity(entity_type: str, value: str, *, source: str = "regex") -> dict[str, str]:
    output_type = _ENTITY_TYPE_TO_OUTPUT.get(entity_type, entity_type)
    item = {"type": output_type, "value": value, "source": source}
    if output_type == "url":
        domain = _url_domain(value)
        if domain:
            item["domain"] = domain
    return item


def extract_regex_entities(text: str) -> list[dict[str, str]]:
    urls = [_clean_regex_value(url) for url in URL_RE.findall(text)]

    # Remove URLs antes de capturar domínios soltos, para não duplicar hosts de links.
    text_no_urls = URL_RE.sub(" ", text)

    entities: list[dict[str, str]] = []

    def add(entity_type: str, values: list[str], validator=None) -> None:
        seen = set()
        for value in values:
            value = _clean_regex_value(value)
            if validator and not validator(value):
                continue
            key = (entity_type, value.lower())
            if value and key not in seen:
                seen.add(key)
                entities.append(_entity(entity_type, value))

    add("url", urls)
    add("email", EMAIL_RE.findall(text))
    add("ip", IP_RE.findall(text))
    add("cve", CVE_RE.findall(text))
    add("hash", HASH_RE.findall(text))
    add("wallet", ETH_RE.findall(text) + BTC_RE.findall(text))
    add("handle", HANDLE_RE.findall(text))
    add("domain", DOMAIN_RE.findall(text_no_urls), _looks_like_domain)
    add("domain", [_refang_domain(value) for value in DEFANGED_DOMAIN_RE.findall(text_no_urls)], _looks_like_domain)

    return entities


def extract_primary_source_url(text: str) -> str:
    for url in URL_RE.findall(text):
        valid = _valid_source_url(url)
        if valid:
            return valid
    return ""


def _is_social_domain(domain: str) -> bool:
    domain = _strip_www(domain)
    return any(domain == d or domain.endswith("." + d) for d in _SOCIAL_DOMAINS)


def _is_likely_ioc_entity(entity: dict[str, str], primary_source_url: str) -> bool:
    entity_type = entity.get("type", "")
    value = entity.get("value", "")
    if entity_type not in _IOC_ENTITY_TYPES:
        return False
    if entity_type in {"cve", "hash", "ip_address", "email", "wallet"}:
        return True
    if entity_type == "url":
        domain = entity.get("domain", "") or _url_domain(value)
        return bool(domain) and not _is_social_domain(domain) and value != primary_source_url
    if entity_type == "domain":
        primary_domain = _url_domain(primary_source_url) if primary_source_url else ""
        return bool(value) and not _is_social_domain(value) and value.lower() != primary_domain
    return False


_BAD_CONTEXT_VALUES = {
    "n/a", "na", "none", "null", "unknown", "not specified", "not applicable",
    "generic", "informational", "low", "medium", "high", "critical",
    "i", "l", "m", "h", "c", "u", "i|l|m|h|c", "l|m|h", "ind|org|gov|ci|u",
}

_GENERIC_VICTIM_TERMS = {
    "advisory", "affected", "article", "bootcamp", "community", "consumers",
    "generic", "owners", "readers", "users", "vulnerability", "workshop",
}

_PRODUCT_HINTS = {
    "android", "apache", "app", "chrome", "cups", "driver", "firefox", "ios",
    "kernel", "linux", "macos", "plugin", "router", "safari", "server", "sharepoint",
    "solana web3.js", "tools", "windows", "wordpress",
}

_PROMOTIONAL_NAMES = {
    "hackingblogs", "hackingblogs.com", "api-hacking bootcamp", "hackingblogs community",
}

_GOV_HINTS = {
    "agency", "court", "department", "federal", "government", "ministry", "national",
    "police", "treasury",
}


def _clean_context_name(value: str) -> str:
    value = re.sub(r"\s+", " ", value.strip())
    return value.strip(" .,;:-")


def _valid_context_name(value: str) -> bool:
    normalized = _clean_context_name(value)
    lowered = normalized.lower()
    if not normalized or lowered in _BAD_CONTEXT_VALUES:
        return False
    if "|" in normalized or "{" in normalized or "}" in normalized:
        return False
    if re.fullmatch(r"[a-z](?:\|[a-z])+", lowered):
        return False
    if CVE_RE.search(normalized):
        return False
    if normalized.islower() and len(normalized) <= 3:
        return False
    if len(normalized) > 96 or len(normalized.split()) > 8:
        return False
    return True


def _looks_promotional_or_generic(name: str) -> bool:
    lowered = name.lower()
    if lowered in _PROMOTIONAL_NAMES or any(p in lowered for p in _PROMOTIONAL_NAMES):
        return True
    tokens = set(re.findall(r"[a-z0-9]+", lowered))
    return bool(tokens & _GENERIC_VICTIM_TERMS)


def _looks_like_product(name: str) -> bool:
    lowered = name.lower()
    if "." in lowered and " " not in lowered:
        return True
    return any(hint in lowered for hint in _PRODUCT_HINTS)


def _looks_like_malware_or_tool(name: str) -> bool:
    lowered = name.lower()
    return any(term in lowered for term in {"keylogger", "malware", "ransomware", "stealer"})


def _infer_victim_type(name: str, target_type: str) -> str:
    if target_type != "unknown":
        return target_type
    lowered = name.lower()
    if any(hint in lowered for hint in _GOV_HINTS):
        return "government"
    return "company"


def _should_keep_victim(name: str, target_type: str, threat_label: str) -> bool:
    if not _valid_context_name(name):
        return False
    if any(sep in name for sep in [",", ";"]):
        return False
    if _looks_like_domain(name):
        return False
    if _looks_promotional_or_generic(name):
        return False
    if _looks_like_malware_or_tool(name):
        return False
    if threat_label == "vulnerability_or_exploit" and (target_type == "unknown" or _looks_like_product(name)):
        return False
    return True


def _add_entity_once(
    entities: list[dict[str, str]],
    entity_type: str,
    value: str,
    source: str = "llm",
    **extra: str,
) -> None:
    value = _clean_context_name(value)
    if not _valid_context_name(value):
        return
    output_type = _ENTITY_TYPE_TO_OUTPUT.get(entity_type, entity_type)
    key = (output_type, value.lower())
    if all((ent.get("type"), ent.get("value", "").lower()) != key for ent in entities):
        ent = _entity(entity_type, value, source=source)
        ent.update({k: v for k, v in extra.items() if v})
        entities.append(ent)


def _demote_non_victim_context(
    victim_name: str,
    target_type: str,
    threat_label: str,
    entities: list[dict[str, str]],
) -> None:
    name = _clean_context_name(victim_name)
    if not _valid_context_name(name) or _should_keep_victim(name, target_type, threat_label):
        return
    if threat_label == "vulnerability_or_exploit" or _looks_like_product(name):
        _add_entity_once(entities, "product", name)
    elif _looks_like_malware_or_tool(name):
        _add_entity_once(entities, "malware", name)


def _structured_name_entities(
    entities: list[dict[str, str]],
    entity_type: str,
    output_type: str,
    threat_label: str,
) -> list[dict[str, str]]:
    values: list[dict[str, str]] = []
    seen = set()
    for ent in entities:
        if ent.get("type") != entity_type:
            continue
        name = _clean_context_name(ent.get("value", ""))
        if not _should_keep_victim(name, output_type, threat_label):
            continue
        key = name.lower()
        if key not in seen:
            seen.add(key)
            values.append({"name": name, "type": _infer_victim_type(name, output_type), "source": ent.get("source", "llm")})
    return values


def _build_victims(
    victim_name: str,
    target_type: str,
    threat_label: str,
    entities: list[dict[str, str]],
) -> list[dict[str, str]]:
    victims = _structured_name_entities(entities, "victim", target_type, threat_label)
    primary_name = _clean_context_name(victim_name)
    if _should_keep_victim(primary_name, target_type, threat_label) and all(v["name"].lower() != primary_name.lower() for v in victims):
        victims.insert(0, {"name": primary_name, "type": _infer_victim_type(primary_name, target_type), "source": "llm"})
    return victims


def _sync_victims_to_entities(victims: list[dict[str, str]], entities: list[dict[str, str]]) -> None:
    for victim in victims:
        _add_entity_once(
            entities,
            "victim",
            victim.get("name", ""),
            source=victim.get("source", "llm"),
            victim_type=victim.get("type", ""),
        )


def valid_incident_date(value: str) -> str:
    value = value.strip()
    return value if re.fullmatch(r"\d{4}-\d{2}-\d{2}", value) else ""

In [5]:
def normalize_compact_result(
    result: CompactThreatClassification,
    post: dict[str, Any],
) -> dict[str, Any]:
    cleaned_text = str(
        post.get("cleaning", {}).get("normalized_message")
        or post.get("message", "")
        or ""
    )
    original_text = str(post.get("message") or cleaned_text or "")

    primary_source_url = extract_primary_source_url(original_text)
    regex_entities = extract_regex_entities(original_text)
    llm_entities = [
        _entity(ent.t, ent.v.strip(), source="llm")
        for ent in result.e
        if ent.v.strip()
    ]

    entities = []
    seen = set()
    for ent in regex_entities + llm_entities:
        key = (ent["type"], ent["value"].lower())
        if key not in seen:
            seen.add(key)
            entities.append(ent)

    ioc_types = sorted({
        ent["type"]
        for ent in entities
        if _is_likely_ioc_entity(ent, primary_source_url)
    })

    threat_label = THREAT_CODE_TO_LABEL.get(result.tl, "other_cyber")
    is_threat = result.r and threat_label != "not_a_threat"

    secondary_labels = [
        THREAT_CODE_TO_LABEL[x]
        for x in dict.fromkeys(result.sl)
        if x in THREAT_CODE_TO_LABEL and x != result.tl
    ]
    target_type = TARGET_CODE_TO_LABEL.get(result.tt, "unknown")
    actor_name = _clean_context_name(result.a.strip())
    actor_type = ACTOR_TYPE_CODE_TO_LABEL.get(result.at, "unknown")
    if _valid_context_name(actor_name):
        _add_entity_once(
            entities,
            "actor",
            actor_name,
            actor_type="" if actor_type == "unknown" else actor_type,
        )
    _demote_non_victim_context(result.v, target_type, threat_label, entities)

    if not is_threat:
        return {
            "cleaned_text": cleaned_text,
            "is_cyber_relevant": False,
            "threat_label": "not_a_threat",
            "secondary_labels": [],
            "severity": "informational",
            "confidence": CONF_CODE_TO_LABEL.get(result.cf, "low"),
            "target_type": "unknown",
            "ioc_present": bool(ioc_types),
            "ioc_types": ioc_types,
            "requires_human_review": result.hr,
            "victim_name": "",
            "victims": [],
            "primary_source_url": primary_source_url,
            "incident_date": "",
            "entities": entities,
        }

    victims = _build_victims(result.v, target_type, threat_label, entities)
    _sync_victims_to_entities(victims, entities)
    victim_name = victims[0]["name"] if victims else ""
    incident_date = valid_incident_date(result.d)

    requires_review = (
        result.hr
        or CONF_CODE_TO_LABEL.get(result.cf, "low") == "low"
        or not victim_name and threat_label in {
            "ransomware",
            "data_breach_or_leak",
            "supply_chain_compromise",
            "account_takeover",
        }
    )

    return {
        "cleaned_text": cleaned_text,
        "is_cyber_relevant": True,
        "threat_label": threat_label,
        "secondary_labels": secondary_labels,
        "severity": SEV_CODE_TO_LABEL.get(result.sv, "informational"),
        "confidence": CONF_CODE_TO_LABEL.get(result.cf, "low"),
        "target_type": target_type,
        "ioc_present": bool(ioc_types),
        "ioc_types": ioc_types,
        "requires_human_review": requires_review,
        "victim_name": victim_name,
        "victims": victims,
        "primary_source_url": primary_source_url,
        "incident_date": incident_date,
        "entities": entities,
    }

In [6]:
records = []
for json_file in JSON_FILES:
    with json_file.open("r", encoding="utf-8") as file:
        payload = json.load(file)
    if isinstance(payload, list):
        for item in payload:
            records.append({"source_file": json_file.name, **item})

df = pd.DataFrame(records)
df.head()

,source_file,date,message,id,cleaning,_id
0,HackingBlogsGroup.json,2025-06-19,🔥FREE NOTES API-HACKING BOOTCAMP DAY 1 : SETTI...,de9562f6-81dc-4df7-9250-d406b1b41a1a,{'normalized_message': 'free notes api-hacking...,NaN
1,HackingBlogsGroup.json,2025-06-09,🚨🚨A Secret Hacker GangExposed Is Exposing the ...,f80e9944-6e76-4d28-ba13-94752755e410,{'normalized_message': 'a secret hacker gangex...,NaN
2,HackingBlogsGroup.json,2025-06-08,🚨Delete These 20 Google Play Apps RIGHT NOW – ...,f77f1360-426c-496e-bc61-acd4fa01c6bb,{'normalized_message': 'delete these 20 google...,NaN
3,HackingBlogsGroup.json,2025-05-29,"🔥 364,000 Americans’ Data Exposed in LexisNexi...",7de299b3-81be-417c-a7e1-a0197d03db8a,{'normalized_message': '364 000 americans data...,NaN
4,HackingBlogsGroup.json,2025-05-24,⚠️ WARNING: TikTok Videos Offering Free Softwa...,67ac4da2-c4fb-4244-893f-e701d1d03f80,{'normalized_message': 'warning tiktok videos ...,NaN


In [7]:
def _load_existing_results() -> tuple[dict[str, list[dict[str, Any]]], set[tuple[str, str]]]:
    existing_by_file: dict[str, list[dict[str, Any]]] = {}
    seen_keys: set[tuple[str, str]] = set()

    for output_path in OUTPUT_DIR.glob("*.json"):
        try:
            with output_path.open("r", encoding="utf-8") as file:
                payload = json.load(file)
        except Exception:
            continue

        if not isinstance(payload, list):
            continue

        file_name = output_path.name
        existing_by_file[file_name] = payload

        for item in payload:
            if not isinstance(item, dict):
                continue
            post_id = item.get("id")
            if post_id is not None:
                seen_keys.add((file_name, str(post_id)))

    return existing_by_file, seen_keys


def _persist_results_for_file(file_name: str, items: list[dict[str, Any]]) -> None:
    output_path = OUTPUT_DIR / file_name
    with output_path.open("w", encoding="utf-8") as file:
        json.dump(items, file, ensure_ascii=False, indent=2)
        file.write("\n")


def _record_key(post: dict[str, Any]) -> tuple[str, str] | None:
    source_file = post.get("source_file")
    post_id = post.get("id")
    if source_file is None or post_id is None:
        return None
    return str(source_file), str(post_id)


def fallback_classification(post: dict[str, Any], exc: Exception) -> dict[str, Any]:
    fallback = CompactThreatClassification(r=False, tl="n", cf="l", hr=True)
    classification = normalize_compact_result(fallback, post)
    classification["classification_error"] = str(exc)
    return classification


async def classify_post(model, post: dict[str, Any]) -> dict[str, Any]:
    prompt = build_prompt(post)
    try:
        result = await model.respond(
            prompt,
            config={
                "systemPrompt": SYSTEM_PROMPT,
                "reasoningEffort": "medium"
                },
            response_format=CompactThreatClassification,
        )
        compact = CompactThreatClassification.model_validate(result.parsed)
        classification = normalize_compact_result(compact, post)
    except Exception as exc:
        classification = fallback_classification(post, exc)

    enriched = dict(post)
    enriched["classification"] = classification
    return enriched


async def classify_all(records: list[dict[str, Any]], max_concurrent: int = MAX_CONCURRENT) -> list[dict[str, Any]]:
    write_lock = asyncio.Lock()

    by_file, seen_keys = _load_existing_results()
    to_process: list[dict[str, Any]] = []

    for post in records:
        key = _record_key(post)
        if key is not None and key in seen_keys:
            continue
        to_process.append(post)

    if not to_process:
        print("Nenhum novo post para classificar (todos ja processados no output).")
        merged_results: list[dict[str, Any]] = []
        for items in by_file.values():
            merged_results.extend(items)
        return merged_results

    queue: asyncio.Queue[dict[str, Any] | None] = asyncio.Queue()
    for post in to_process:
        queue.put_nowait(post)

    for _ in range(max_concurrent):
        queue.put_nowait(None)

    processed_count = 0

    async with lms.AsyncClient(LMS_HOST) as client:
        model = await client.llm.model(MODEL)

        async def worker() -> None:
            nonlocal processed_count
            while True:
                post = await queue.get()
                if post is None:
                    queue.task_done()
                    return

                try:
                    classified = await classify_post(model, post)
                    async with write_lock:
                        source_file = str(classified.get("source_file", "unknown.json"))
                        by_file.setdefault(source_file, []).append(classified)
                        _persist_results_for_file(source_file, by_file[source_file])

                        key = _record_key(classified)
                        if key is not None:
                            seen_keys.add(key)

                        processed_count += 1
                finally:
                    queue.task_done()

        workers = [asyncio.create_task(worker()) for _ in range(max_concurrent)]

        try:
            with tqdm(total=len(to_process)) as progress:
                while processed_count < len(to_process):
                    await asyncio.sleep(0.2)
                    progress.n = processed_count
                    progress.refresh()
        except (asyncio.CancelledError, KeyboardInterrupt):
            for task in workers:
                task.cancel()
            await asyncio.gather(*workers, return_exceptions=True)
            raise

        await queue.join()
        await asyncio.gather(*workers)

    merged_results: list[dict[str, Any]] = []
    for items in by_file.values():
        merged_results.extend(items)

    print(f"Classificados agora: {processed_count} | Total no output: {len(merged_results)}")
    return merged_results

In [8]:
results = await classify_all(records)

by_file: dict[str, list[dict[str, Any]]] = {}
for item in results:
    by_file.setdefault(item["source_file"], []).append(item)

print(f"Done. {len(results)} posts disponiveis em {len(by_file)} arquivos no output.")

100%|██████████| 1/1 [00:04<00:00,  4.04s/it]

Classificados agora: 1 | Total no output: 9932
Done. 9932 posts disponiveis em 10 arquivos no output.


In [9]:
# import os
# import time

# # Optional: add a small delay so you can see the output before shutdown starts
# print("Processing finished. Shutting down Windows in 30 seconds...")
# time.sleep(30)

# # Call Windows shutdown command
# # /s = shutdown, /t 0 = immediately (change to 30 for 30-second delay, etc.)
# os.system("shutdown.exe /s /t 0 /c \"Jupyter notebook requested shutdown\"")

# print("Shutdown command sent.")